# Phase 5 — low-vigor alerts

Runs entirely offline from `data/processed/block_timeseries.parquet` plus
`data/blocks.geojson` (no Earth Engine). Plain statistics — this is the whole
model.

For each block, a **day-of-year baseline** is built from that block's *prior*
seasons: at each observation's day of year, pool prior-season observations
within ±10 days and take median, IQR, and 10th percentile. A block-season is
marked **below baseline** where two consecutive observations fall below that
10th percentile. A **same-variety comparison** in the same season reads
all-down as weather and one-down as a single-block cause.

**Baseline floor:** per-block `baseline_start` from `blocks.geojson` (the
first real post-planting season), so the baseline only ever compares vines to
vines — never to the previous ground cover. Both blocks were planted 2024, so
this is only now (2025–2026) accumulating a real baseline; that's correct, not
a bug.

**Acceptance criteria (Phase 5):** below-10th-percentile for two consecutive
observations raises an alert; the same-variety comparison separates weather
(all down) from a single-block cause (one down).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vigor import vigor_alerts, extract, ingest, plots

PARQUET = PROJECT_ROOT / "data" / "processed" / "block_timeseries.parquet"
ALERTS_PARQUET = PROJECT_ROOT / "data" / "processed" / "vigor_alerts.parquet"
FIGURES = PROJECT_ROOT / "outputs" / "figures"

In [ ]:
ts = ingest.load_timeseries(PARQUET)
blocks = extract.load_blocks(PROJECT_ROOT / "data" / "blocks.geojson")

# Per-block baseline floor: the first real post-planting season.
baseline_start = dict(zip(blocks["block_id"], blocks["baseline_start"].astype(int)))
baseline_start

## Compute

One row per in-season observation with the baseline stats it was judged
against, `below_p10`, and `below_baseline`. Observations with no admissible
prior season have NaN baseline and are never marked.

In [ ]:
al = vigor_alerts.below_baseline_seasons(ts, baseline_start=baseline_start)
vigor_alerts.write_alerts(al, ALERTS_PARQUET)
print(f"alerts -> {ALERTS_PARQUET}")

have_base = al.assign(has_base=al['n_baseline'] >= 8).groupby('year')['has_base'].mean().round(2)
print('\nfraction of observations with a usable baseline, by year:')
print(have_base.to_string())

## Below-baseline events

In [ ]:
events = vigor_alerts.stress_events(al)
events if len(events) else 'No below-baseline events yet — insufficient vine-history baseline.'

## Compare against same-variety blocks

The two blocks are different varieties (chardonnay, merlot) at different
sites, so **each is a variety of one** — the comparison correctly reports
`no peers`, meaning weather vs. a single-block cause can't be told apart from
vigor alone. This table becomes discriminating as soon as a second
same-variety block is added to `blocks.geojson`.

In [ ]:
cc = vigor_alerts.compare_same_variety(al, blocks)
cc if len(cc) else 'No below-baseline seasons to compare yet.'

## Baseline-comparison figures

Per block, per season with a baseline: current-season NDVI (blue) against the
prior-season baseline (gray IQR band + median, orange 10th-percentile line),
with below-baseline runs marked in red.

In [ ]:
for block_id in sorted(al['block_id'].unique()):
    fig = plots.baseline_comparison(al, block_id)
    saved = plots.save_figure(fig, FIGURES / f"05_{block_id}_baseline.png")
    print(f"figure -> {saved}")